In [1]:
!pip install transformers sentencepiece sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.5 MB/s eta 0:00:00


In [2]:
#load pretrained translation model
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/mbart-large-50-many-to-many-mmt"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/529 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

In [3]:
# Translation function
def translate_en_to_hi(text):
    tokenizer.src_lang = "en_XX"

    encoded = tokenizer(text, return_tensors="pt")

    generated_tokens = model.generate(
        **encoded,
        forced_bos_token_id=tokenizer.lang_code_to_id["hi_IN"]
    )

    translated = tokenizer.batch_decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return translated[0]

In [4]:
text = "Government hospitals provide free vaccination."

print(translate_en_to_hi(text))

सरकारी अस्पताल निःशुल्क टीकाकरण उपलब्ध कराते हैं।


In [5]:
import sacrebleu

reference = ["सरकारी अस्पताल मुफ्त टीकाकरण प्रदान करते हैं।"]
candidate = translate_en_to_hi("Government hospitals provide free vaccination.")

bleu = sacrebleu.corpus_bleu([candidate], [reference])
print("BLEU score:", bleu.score)

BLEU score: 15.619699684601283


In [6]:
# New test sentence
test_sentence = "Passengers must carry valid identification during travel."

# Reference translation (human-written)
reference_translation = [
    "यात्रा के दौरान यात्रियों को वैध पहचान पत्र साथ रखना चाहिए।"
]

# Model prediction
predicted_translation = translate_en_to_hi(test_sentence)

print("English:", test_sentence)
print("Model Translation:", predicted_translation)
print("Reference Translation:", reference_translation[0])

English: Passengers must carry valid identification during travel.
Model Translation: यात्रियों को यात्रा के दौरान वैध पहचान लेनी चाहिए।
Reference Translation: यात्रा के दौरान यात्रियों को वैध पहचान पत्र साथ रखना चाहिए।


In [7]:
import sacrebleu

# sacrebleu expects list format
bleu_score = sacrebleu.corpus_bleu(
    [predicted_translation],   # system output
    [reference_translation]    # reference list
)

print("BLEU Score:", bleu_score.score)

BLEU Score: 21.596066896843258


In [8]:
# Multiple test sentences
test_sentences = [
    "Government hospitals provide free vaccination.",
    "Passengers must carry valid identification during travel.",
    "Please wear a mask in crowded places."
]

# Human reference translations
reference_translations = [
    "सरकारी अस्पताल मुफ्त टीकाकरण प्रदान करते हैं।",
    "यात्रा के दौरान यात्रियों को वैध पहचान पत्र साथ रखना चाहिए।",
    "कृपया भीड़भाड़ वाली जगहों पर मास्क पहनें।"
]

In [9]:
# Generate predictions
predictions = []
for sent in test_sentences:
    pred = translate_en_to_hi(sent)
    predictions.append(pred)

# Print results nicely
for i in range(len(test_sentences)):
    print(f"\nSentence {i+1}")
    print("English:", test_sentences[i])
    print("Model:", predictions[i])
    print("Reference:", reference_translations[i])


Sentence 1
English: Government hospitals provide free vaccination.
Model: सरकारी अस्पताल निःशुल्क टीकाकरण उपलब्ध कराते हैं।
Reference: सरकारी अस्पताल मुफ्त टीकाकरण प्रदान करते हैं।

Sentence 2
English: Passengers must carry valid identification during travel.
Model: यात्रियों को यात्रा के दौरान वैध पहचान लेनी चाहिए।
Reference: यात्रा के दौरान यात्रियों को वैध पहचान पत्र साथ रखना चाहिए।

Sentence 3
English: Please wear a mask in crowded places.
Model: भीड़-भाड़ वाले स्थानों में कृपया मास्क पहनें।
Reference: कृपया भीड़भाड़ वाली जगहों पर मास्क पहनें।


In [10]:
# BLEU calculation (IMPORTANT format)
import sacrebleu

bleu = sacrebleu.corpus_bleu(
    predictions,
    [reference_translations]
)

print("\nFinal BLEU Score:", bleu.score)


Final BLEU Score: 13.05283035773873
